# 05 · Socioeconomic

Computes commercial and residential ratios for each establishment using NYC PLUTO tax-lot data.

For each shop, matches to the nearest PLUTO lot via BallTree and computes:
- `com_ratio` = commercial area / total building area
- `res_ratio` = residential area / total building area

**Input:** `csv/00_base_data.csv`, PLUTO CSV
**Output:** `csv/05_socioeconomic.csv` — `osm_id`, `com_ratio`, `res_ratio`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

# PLUTO data path
PLUTO_CSV = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

In [ ]:
# ── Load PLUTO and build BallTree ─────────────────────

print("Loading PLUTO data...")
pluto = pd.read_csv(PLUTO_CSV, low_memory=False)
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()
print(f"  {len(pluto_mn)} Manhattan lots loaded")

# Build spatial index on PLUTO lots
pluto_coords = np.radians(pluto_mn[["latitude", "longitude"]].values)
tree = BallTree(pluto_coords, metric="haversine")

# Query nearest lot for each shop
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

In [ ]:
# ── Compute com_ratio and res_ratio ───────────────────

matched = pluto_mn.iloc[indices.flatten()].copy()

df["com_ratio"] = matched["comarea"].values / matched["bldgarea"].replace(0, np.nan).values
df["res_ratio"] = matched["resarea"].values / matched["bldgarea"].replace(0, np.nan).values

print(f"com_ratio filled : {df['com_ratio'].notna().sum()}/{len(df)}")
print(f"res_ratio filled : {df['res_ratio'].notna().sum()}/{len(df)}")
print(f"\nAvg com_ratio : {df['com_ratio'].mean():.4f}")
print(f"Avg res_ratio : {df['res_ratio'].mean():.4f}")

In [ ]:
df_out = df[["osm_id", "com_ratio", "res_ratio"]]
df_out.to_csv("csv/05_socioeconomic.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/05_socioeconomic.csv")
print(df_out.describe().round(4))